# RAG Pipeline - Exp1

* The goal is to finish multiple RAG pipelines in fastest way and accurately 🎯
* Compare LlamaIndex RAG performance in different settings
* Save the configs file for each pipeline

In [7]:
%load_ext autoreload
%autoreload 2

import os
import yaml
os.environ.pop("OPENAI_API_KEY", None)

from datasets import load_dataset
import pandas as pd
import json

from utils import *

import warnings
warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Testing Dataset

In [4]:
# Load raw data with smalll selected testset
dataset = load_dataset("PatronusAI/financebench")
selected_items = dataset['train'].select(range(5))

selected_docs =set()
for dct in selected_items:
    selected_docs.add(f'pdfs/{dct["doc_name"]}.pdf')
print(len(selected_docs))

# Load selected pdfs
loaded_pdf = {}
with ThreadPoolExecutor(max_workers=5) as executor:
    loaded_pdf = dict(executor.map(load_pdf_content, list(selected_docs)))
print(f'{len(loaded_pdf)} pdfs loaded')


# Genrate LlamaIndex Documents from each selected item
documents = []
selected_metadata_cols = ['company', 'doc_name', 'doc_period', 'doc_link']

documents = process_items_parallel(
    selected_items,
    selected_doc_names=set([selected_doc.replace('pdfs/', '').replace('.pdf', '')
                             for selected_doc in selected_docs]),
    loaded_pdf=loaded_pdf,
    selected_metadata_cols=selected_metadata_cols,
    max_workers=10
)
print(f"Generated {len(documents)} documents")

2
2 pdfs loaded
Generated 5 documents


### Run Pipeline with Different Settings

#### Split Config File --> Per Model Per Config

In [5]:
with open("all_rag_pipeline_config.yaml", "r", encoding="utf-8") as f:
     all_config= yaml.safe_load(f)

retriever_params = all_config['retriever_params']
cfgs = []
for i in range(len(all_config['yaml_config_files'])-1):
    yaml_config_file = 'output/saved_configs/' + all_config['yaml_config_files'][i]
    cfgs.append(yaml_config_file)
    config = {
        'llm_model': all_config['llm_models'][i],
        'embedding_model': all_config['embedding_models'][i],
        'indexing_storage_dir': all_config['indexing_storage_dirs'][i],
        'output_file': all_config['output_files'][i],
        'retriever_params': retriever_params
    }

    with open(yaml_config_file, "w", encoding="utf-8") as f:
        yaml.safe_dump(config, f, sort_keys=False)

#### Sequentially Run Embedding & Saving Indexing

In [8]:
for i in range(len(all_config['embedding_models'])):
    embed_model_str = all_config['embedding_models'][i]
    indexing_storage_dir = all_config['indexing_storage_dirs'][i]
    print(f"Embedding model: {embed_model_str}")
    get_vector_index(documents, embed_model_str, indexing_storage_dir)
    

Embedding model: BAAI/bge-small-en-v1.5


Parsing nodes:   0%|          | 0/5 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1191 [00:00<?, ?it/s]

Index saved to ./storage_m1
Embedding model: thenlper/gte-large


Parsing nodes:   0%|          | 0/5 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Parsing nodes:   0%|          | 0/5 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/1191 [00:00<?, ?it/s]

Index saved to ./storage_m3
